# Underwater Fish Detection — training on Google Colab
Runtime → Change runtime type → **T4 GPU**. Upload this project folder (without `.venv`, `data`, `runs`) to Drive or clone it, then run the cells.

In [ ]:
!nvidia-smi
%cd /content/underwater-fish-detection
!pip -q install ultralytics imageio-ffmpeg roboflow

## 1. Data
Public RF100 mirror of the Roboflow Aquarium dataset (no API key). To pull from Roboflow instead, set `ROBOFLOW_API_KEY` and use `--download roboflow`.

In [ ]:
!python scripts/prepare_dataset.py --download hf --preprocess none --aug-copies 1 --out data/yolo
!python scripts/prepare_dataset.py --preprocess clahe --aug-copies 1 --out data/yolo_clahe

## 2. Train

In [ ]:
!python scripts/train.py --data data/yolo/data.yaml --model yolov8s.pt --epochs 50 --name y8s_raw
!python scripts/train.py --data data/yolo_clahe/data.yaml --model yolov8s.pt --epochs 50 --name y8s_clahe

## 3. Evaluate (precision, recall, mAP50, mAP50-95)

In [ ]:
!python scripts/evaluate.py y8s_raw:data/yolo/data.yaml y8s_clahe:data/yolo_clahe/data.yaml

## 4. Video inference

In [ ]:
!python scripts/video_infer.py --source data/videos/genoa_aquarium_15s.mp4 --weights models/y8s_raw.pt

## 5. Download the weights
Copy `models/*.pt` and `results/` back to your machine, then run `streamlit run app.py` locally.

In [ ]:
from google.colab import files
for f in ['models/y8s_raw.pt', 'models/y8s_clahe.pt', 'results/metrics.json']:
    files.download(f)